In [ ]:
import numpy as np
import copy
import pandas as pd
import torch 
import torch.nn as nn 
import os 
import re
from torch.utils.data import Dataset, DataLoader 
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
from collections import Counter

In [ ]:
torch.manual_seed(42)

In [ ]:
ROOT_DIR = os.path.abspath(os.path.join(os.path.join(os.getcwd(), ".."), ".."))
DATA_DIR = os.path.join(ROOT_DIR, "Data")
DATASET_DIR = os.path.join(DATA_DIR, "SMS-Spam-Data")
DATASET_FILE = os.path.join(DATASET_DIR, "Dataset_5971.csv")


In [ ]:
df = pd.read_csv(DATASET_FILE)
df.head()

In [ ]:
tab_mask = df["TEXT"].str.startswith("\t")
print("Rows with leading \\t artifact:", tab_mask.sum())
print(df.loc[tab_mask, "LABEL"].value_counts())

df["TEXT"] = df["TEXT"].str.lstrip("\t")
assert not df["TEXT"].str.startswith("\t").any()

In [ ]:
df.duplicated().sum(), df.shape

In [ ]:
df["CLEAN"] = df["TEXT"].str.lower().str.split().str.join(" ")
df["LABEL"] = df["LABEL"].str.strip().str.lower()
df.head()

In [ ]:
df = df.drop_duplicates(subset="CLEAN").drop(columns="CLEAN")
df.shape, df.duplicated().sum()

In [ ]:
URL_RE = re.compile(
    r"(https?://\S+|www\.\S+|\b[a-zA-Z0-9-]+\.(?:com|co\.uk|org\.uk|org|net|info|biz)(?:/\S*)?)",
    re.IGNORECASE,
)
PHONE_RE = re.compile(r"(\+?\d[\d\-\s]{6,}\d)")

def mask_text(text: str) -> str:
    text = URL_RE.sub("<URL>", text)
    text = PHONE_RE.sub("<PHONE>", text)
    return text

df["TEXT_MASKED"] = df["TEXT"].apply(mask_text)

n_url = (df["TEXT"] != df["TEXT"].apply(lambda t: URL_RE.sub("<URL>", t))).sum()
n_phone = (df["TEXT"] != df["TEXT"].apply(lambda t: PHONE_RE.sub("<PHONE>", t))).sum()
print(f"Rows with a <URL> substitution:   {n_url}")
print(f"Rows with a <PHONE> substitution: {n_phone}")

df.loc[df["TEXT"] != df["TEXT_MASKED"], ["TEXT", "TEXT_MASKED"]].head(5)

In [ ]:
X = df["TEXT_MASKED"]
y = df["LABEL"]

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1, stratify=y_temp, random_state=42)


In [ ]:
encoder = LabelEncoder()

y_train_encoded = encoder.fit_transform(y_train).tolist()
y_val_encoded = encoder.transform(y_val).tolist()
y_test_encoded = encoder.transform(y_test).tolist()

print(len(set(y_train_encoded)), len(set(y_test_encoded)), len(set(y_val_encoded)))

type(y_train_encoded)

# Tokenization for X_temp, X_train, X_val
Simple tokenization by splitting between whitespaces

In [ ]:
X_train_tokenized = X_train.str.split()
X_val_tokenized = X_val.str.split()
X_test_tokenized = X_test.str.split()

In [ ]:
print(X_train_tokenized.iloc[0])

# Vocabulary building

In [ ]:
# Built only from the training corpus tokens, unseen on test and validation tokens 
token_frequency_map = Counter(X_train_tokenized.explode()) # .explode().value_counts() is also an alternate option
print(token_frequency_map)

In [ ]:
filtered_tokens = [token for token,count in token_frequency_map.most_common() if count >= 2]
index_to_token = ["PAD", "UNK"] + filtered_tokens
token_to_index = {token: index for index, token in enumerate(index_to_token)}

# Creating embeddings 

In [ ]:
def create_embedding(pd_series):
    data = []
    for i in range(len(pd_series)):
        current_row = pd_series.iloc[i]
        embedding = [token_to_index.get(current_row[j], 1) for j in range(len(current_row))]
        data.append(embedding)
    return data

X_train_embedding = create_embedding(X_train_tokenized)
X_test_embedding = create_embedding(X_test_tokenized)
X_val_embedding = create_embedding(X_val_tokenized)




In [ ]:
train_lengths = X_train_tokenized.apply(len)          # one integer per message
print(train_lengths.describe())          # mean, median, max
max_len = int(np.percentile(train_lengths, 95))
print("Candidate: ", max_len) 

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, X, y, max_len):
        self.X = X
        self.y = y
        self.max_len = max_len

    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx][:self.max_len], self.y[idx]
    

In [ ]:
train_dataset = CustomDataset(X_train_embedding, y_train_encoded, max_len)
val_dataset = CustomDataset(X_val_embedding, y_val_encoded, max_len)
test_dataset = CustomDataset(X_test_embedding, y_test_encoded, max_len)


In [ ]:
def collate_fn(batch):
    ids, labels = zip(*batch)                                   # unpack the (ids, label) pairs
    lengths = torch.tensor([len(x) for x in ids], dtype=torch.long)
    padded = pad_sequence(
        [torch.tensor(x, dtype=torch.long) for x in ids],
        batch_first=True,                                       # shape (batch, longest)
        padding_value=0,                                        # <PAD> index
    )
    labels = torch.tensor(labels, dtype=torch.long)
    return padded, lengths, labels

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)


In [ ]:
padded, lengths, labels = next(iter(train_loader))
print(padded.shape, padded.dtype, lengths[:5], labels.dtype)

In [ ]:
class ClassificationModel(nn.Module):
    def __init__(self, vocab_size=len(index_to_token), num_labels=3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 64, padding_idx=0)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(64, num_labels)

    def forward(self, padded):
        emb = self.embedding(padded)                 # (B, L, D)         e.g. (32, 30, 64)
        mask = (padded != 0).unsqueeze(-1).float()   # (B, L, 1)         1.0 for real tokens, 0.0 for padding

        summed = (emb * mask).sum(dim=1)             # (B, D)            add up the real tokens' vectors
        counts = mask.sum(dim=1).clamp(min=1)        # (B, 1)            how many real tokens per message
        pooled = summed / counts                     # (B, D)            the average

        pooled = self.dropout(pooled)
        return self.fc(pooled) 

In [ ]:
model = ClassificationModel(len(index_to_token), 3)

In [ ]:
counts = torch.bincount(torch.tensor(y_train_encoded), minlength=3).float()
class_weights = counts.sum() / (3 * counts)
loss_fn = nn.CrossEntropyLoss(weight=class_weights)

In [ ]:
epochs = 50 
learning_rate = 3e-3
optimizer = torch.optim.Adam(params=model.parameters(), lr=learning_rate)

In [ ]:

best_f1, best_state = 0, None

for epoch in range(epochs):
    total_loss = 0
    model.train()
    for X_batch, _, y_batch in train_loader:
        y_pred = model(X_batch)

        loss = loss_fn(y_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    
    model.eval()
    preds, trues = [], []
    with torch.inference_mode():
        correct = 0
        total = 0
        for X_batch, _, y_batch in val_loader:
            y_pred = model(X_batch)
            preds.extend(y_pred.argmax(dim=1).tolist())
            trues.extend(y_batch.tolist())
        
    val_f1 = f1_score(trues, preds, average="macro")
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = copy.deepcopy(model.state_dict())

    print(f"Epoch {epoch+1}/{epochs}  loss {avg_loss:.4f}  val macro-F1 {val_f1:.4f}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from scipy.sparse import csr_matrix

In [ ]:

model.load_state_dict(best_state)   # best epoch's weights, not the last epoch's
model.eval()

preds, trues = [], []
with torch.inference_mode():
    for X_batch, _, y_batch in test_loader:
        logits = model(X_batch)
        preds.extend(logits.argmax(dim=1).tolist())
        trues.extend(y_batch.tolist())

print(classification_report(trues, preds, target_names=encoder.classes_, digits=3))
print(confusion_matrix(trues, preds))

# Sklearn Baseline 


In [ ]:
def to_bow_matrix(embeddings, vocab_size):
    rows, cols, data = [], [], []
    for i, seq in enumerate(embeddings):
        for token_idx, count in Counter(seq).items():
            rows.append(i)
            cols.append(token_idx)
            data.append(count)
    return csr_matrix((data, (rows, cols)), shape=(len(embeddings), vocab_size))

vocab_size = len(index_to_token)
X_train_bow = to_bow_matrix(X_train_embedding, vocab_size)
X_test_bow = to_bow_matrix(X_test_embedding, vocab_size)
X_val_bow = to_bow_matrix(X_val_embedding, vocab_size)

model2 = LogisticRegression(class_weight="balanced", max_iter=1000)
model2.fit(X_train_bow, y_train_encoded)

In [ ]:
y_pred_val = model2.predict(X_val_bow)
report = classification_report(y_val_encoded, y_pred_val, target_names=encoder.classes_, zero_division=0)
print(report)